# Naive Bayes com a base de Gripe


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)


In [ ]:

# Link do Google Sheets em formato CSV
url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv&gid=282951434"

df = pd.read_csv(url)

mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe_ano_passado",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes_cheios",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia",
    "Quantas horas você dormiu em média por noite no ano passado?": "horas_sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem_maos",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

df = df.rename(columns=mapping)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()

display(df.head())
print(df.shape)


,timestamp,gripe_ano_passado,vacina,ambientes_cheios,viajou,alergia,horas_sono,exercicio,alimentacao,lavagem_maos,estresse
0,24/03/2026 15:01:35,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,24/03/2026 15:04:20,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,24/03/2026 15:04:20,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0
3,24/03/2026 15:04:37,Sim,Não,Não,Nuca,Muito,mais de 6 horas,Sim,Às vezes,2 vezes ou menos,2.0
4,24/03/2026 15:05:27,Sim,Sim,Sim,Pelo menos uma vez por mês,Médio,entre 4 e 6 horas,Não,Às vezes,6 a 10 vezes,4.0


(186, 11)


In [ ]:

url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv&gid=282951434"

df = pd.read_csv(url)

mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe_ano_passado",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes_cheios",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia", # Corrigido: adicionados os espaços finais
    "Quantas horas você dormiu em média por noite no ano passado?": "horas_sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem_maos",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

df = df.rename(columns=mapping)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()


df["gripe_ano_passado"] = df["gripe_ano_passado"].str.lower()
df["vacina"] = df["vacina"].str.lower()
df["ambientes_cheios"] = df["ambientes_cheios"].str.lower()
df["viajou"] = df["viajou"].replace({"Nuca": "Nunca", "nuca": "nunca"}).str.lower()
df["exercicio"] = df["exercicio"].str.lower()
df["alimentacao"] = df["alimentacao"].str.lower()
df["lavagem_maos"] = df["lavagem_maos"].str.lower()
df["alergia"] = df["alergia"].str.lower()
df["horas_sono"] = df["horas_sono"].str.lower()


df["estresse"] = pd.to_numeric(df["estresse"], errors="coerce")
map_estresse = {1.0: "muito_baixo", 2.0: "baixo", 3.0: "medio", 4.0: "alto", 5.0: "muito_alto"}
df["estresse"] = df["estresse"].map(map_estresse)

print("Valores por coluna:")
for col in df.columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

Valores por coluna:

timestamp
timestamp
24/03/2026 15:04:20    2
25/03/2026 21:54:31    2
25/03/2026 21:53:55    2
24/03/2026 15:05:27    1
24/03/2026 15:05:29    1
                      ..
26/03/2026 18:16:34    1
27/03/2026 08:47:46    1
27/03/2026 08:55:55    1
27/03/2026 10:00:50    1
29/03/2026 16:35:07    1
Name: count, Length: 183, dtype: int64

gripe_ano_passado
gripe_ano_passado
sim    109
não     77
Name: count, dtype: int64

vacina
vacina
sim    99
não    87
Name: count, dtype: int64

ambientes_cheios
ambientes_cheios
sim    161
não     25
Name: count, dtype: int64

viajou
viajou
poucas vezes por ano          121
pelo menos uma vez por mês     49
nunca                          16
Name: count, dtype: int64

alergia
alergia
não      69
médio    49
pouco    46
muito    22
Name: count, dtype: int64

horas_sono
horas_sono
mais de 6 horas      103
entre 4 e 6 horas     79
4 horas ou menos       4
Name: count, dtype: int64

exercicio
exercicio
sim    128
não     58
Name: count, dt

In [ ]:


features = [
    "vacina",
    "ambientes_cheios",
    "viajou",
    "alergia",
    "horas_sono",
    "exercicio",
    "alimentacao",
    "lavagem_maos",
    "estresse"
]

target = "gripe_ano_passado"

X = df[features].copy()
y = df[target].copy()

print("Distribuição da classe alvo:")
print(y.value_counts())


Distribuição da classe alvo:
gripe_ano_passado
sim    109
não     77
Name: count, dtype: int64


In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


modelo = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ("nb", CategoricalNB())
])

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))


Acurácia: 0.6052631578947368

Matriz de confusão:
[[ 4 12]
 [ 3 19]]

Relatório de classificação:
              precision    recall  f1-score   support

         não       0.57      0.25      0.35        16
         sim       0.61      0.86      0.72        22

    accuracy                           0.61        38
   macro avg       0.59      0.56      0.53        38
weighted avg       0.60      0.61      0.56        38



In [ ]:


resultado = X_test.copy()
resultado["real"] = y_test.values
resultado["previsto"] = y_pred

display(resultado.head(20))


,vacina,ambientes_cheios,viajou,alergia,horas_sono,exercicio,alimentacao,lavagem_maos,estresse,real,previsto
137,sim,sim,poucas vezes por ano,pouco,mais de 6 horas,sim,às vezes,6 a 10 vezes,medio,sim,sim
57,não,sim,pelo menos uma vez por mês,muito,mais de 6 horas,sim,"sim, a maior parte do tempo",3 a 5 vezes,alto,não,sim
100,sim,sim,poucas vezes por ano,não,mais de 6 horas,não,"sim, a maior parte do tempo",6 a 10 vezes,medio,não,sim
171,sim,não,poucas vezes por ano,não,mais de 6 horas,não,às vezes,6 a 10 vezes,medio,não,sim
133,sim,sim,poucas vezes por ano,não,entre 4 e 6 horas,não,"sim, a maior parte do tempo",6 a 10 vezes,muito_alto,sim,sim
126,não,sim,pelo menos uma vez por mês,não,mais de 6 horas,não,às vezes,3 a 5 vezes,muito_alto,não,sim
40,sim,sim,poucas vezes por ano,pouco,entre 4 e 6 horas,sim,"sim, a maior parte do tempo",3 a 5 vezes,alto,sim,sim
68,sim,sim,pelo menos uma vez por mês,muito,mais de 6 horas,sim,"sim, a maior parte do tempo",3 a 5 vezes,muito_alto,sim,sim
149,sim,sim,pelo menos uma vez por mês,não,mais de 6 horas,sim,às vezes,2 vezes ou menos,medio,não,não
155,sim,não,poucas vezes por ano,não,mais de 6 horas,sim,"sim, a maior parte do tempo",2 vezes ou menos,alto,sim,sim


In [ ]:


novo_paciente = pd.DataFrame([{
    "vacina": "sim",
    "ambientes_cheios": "sim",
    "viajou": "poucas vezes por ano",
    "alergia": "pouco",
    "horas_sono": "mais de 6 horas",
    "exercicio": "sim",
    "alimentacao": "sim, a maior parte do tempo",
    "lavagem_maos": "6 a 10 vezes",
    "estresse": "medio"
}])

previsao = modelo.predict(novo_paciente)[0]
probs = modelo.predict_proba(novo_paciente)[0]

print("Previsão:", previsao)
for classe, prob in zip(modelo.classes_, probs):
    print(f"{classe}: {prob:.4f}")


Previsão: sim
não: 0.4623
sim: 0.5377
